# Palm92 RepoGuard v0.9: Why did medium-v001 fail?

Your initial three-task run has already been archived. This diagnostic runs **one** invoice task on an isolated copy and saves all model actions plus audit events. It must not overwrite `medium-v001`. Do not publish the audit before reviewing it: it includes model text and code content.

## Step 1: Clean checkout with the diagnostic script

Choose **T4 GPU** in Runtime settings first. If reusing a previous runtime, start in `/content` before replacing the temporary clone.

In [ ]:
%cd /content
!rm -rf /content/palm92-repoguard
!git clone --branch v0.9-hard-benchmarks --single-branch https://github.com/faithfulord1/palm92-repoguard.git /content/palm92-repoguard
%cd /content/palm92-repoguard
!python -m pip install -e '.[gemma,dev]'

## Step 2: Quick safety checks and GPU check

Stop if either the RepoGuard regression tests fail or CUDA is unavailable. The benchmark fixture checker intentionally confirms that unmodified fixtures have failing tests.

In [ ]:
!python -m pytest -q
!python scripts/check_v0_9_fixtures.py
import torch
assert torch.cuda.is_available(), 'CUDA not available. Select Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## Step 3: Run ONE diagnostic task

Six model steps, instead of another full three-task GPU run. This tests whether the agent emits valid actions, proposes changes, or loops. To repeat, select a new diagnostic ID.

In [ ]:
!python scripts/run_v0_9_diagnostic.py --experiment-id diagnostic-v001 --max-steps 6

## Step 4: Inspect the captured event types

Do not infer the failure mechanism without checking the audit JSON, especially `model_action`, `invalid_model_json`, and `tool_observation`.

In [ ]:
from pathlib import Path
import json
p=Path('artifacts/v0.9/diagnostic-v001')
print((p/'event-counts.json').read_text())
audit=json.loads((p/'audit.json').read_text())
for e in audit.get('events',[]):
    if e.get('event_type') in ('model_action','invalid_model_json','tool_observation'):
        print(json.dumps(e, ensure_ascii=False)[:1200])

## Step 5: Download the separate diagnostic evidence

The ZIP includes the full model action audit; review before sharing publicly. Upload to this chat for diagnosis.

In [ ]:
from google.colab import files
import shutil
archive=shutil.make_archive('/content/repoguard-v0.9-diagnostic-v001','zip',root_dir='artifacts/v0.9/diagnostic-v001')
files.download(archive)